In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from langchain_chroma import Chroma
except Exception:
    try:
        from langchain_community.vectorstores import Chroma  # type: ignore
    except Exception:
        Chroma = None
        print("Chroma is not installed. Install langchain-chroma or langchain-community to build the vector index.")

try:
    from langchain_community.embeddings import SentenceTransformerEmbeddings
except Exception:
    try:
        from langchain.embeddings import SentenceTransformerEmbeddings  # type: ignore
    except Exception:
        SentenceTransformerEmbeddings = None
        print("SentenceTransformerEmbeddings is not installed. Install sentence-transformers/langchain embedding support to build the vector index.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

@dataclass
class VectorConfig:
    vector_records_path: Path = ROOT / "data" / "processed_reports" / "semantic_vector_records.parquet"
    chroma_dir: Path = ROOT / "storage" / "chroma_semantic"
    chroma_collection: str = "insightviewer_semantic"
    embed_model: str = os.getenv("EMBED_MODEL", "all-MiniLM-L12-v2")

CFG = VectorConfig()
CFG.chroma_dir.mkdir(parents=True, exist_ok=True)
print(f"Chroma target: {CFG.chroma_dir}")

## **Load Vector Records**

In [ ]:
def read_dataframe(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix == ".parquet":
            return pd.read_parquet(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)

    csv_fallback = path.with_suffix(".csv")
    if csv_fallback.exists():
        return pd.read_csv(csv_fallback)

    print(f"Missing artifact: {path}")
    return pd.DataFrame()


vector_records_df = read_dataframe(CFG.vector_records_path)
print("semantic vector records:", vector_records_df.shape)
vector_records_df.head(10)

## **Build Chroma Index**

In [ ]:
BUILD_CHROMA_INDEX = True # Set to False to skip building the Chroma index and just read the vector records


def metadata_from_json(value: str) -> dict[str, Any]:
    try:
        meta = json.loads(value)
    except Exception:
        return {}
    return {key: val for key, val in meta.items() if isinstance(val, (str, int, float, bool)) and val is not None}


def build_chroma_index(cfg: VectorConfig, records: pd.DataFrame) -> int:
    if records.empty:
        print("No vector records to index.")
        return 0
    if Chroma is None or SentenceTransformerEmbeddings is None:
        print("Skipping Chroma index because Chroma or embeddings support is not installed.")
        return 0

    embeddings = SentenceTransformerEmbeddings(model_name=cfg.embed_model)
    store = Chroma(
        collection_name=cfg.chroma_collection,
        persist_directory=str(cfg.chroma_dir),
        embedding_function=embeddings,
    )

    texts = records["document"].astype(str).tolist()
    ids = records["vector_id"].astype(str).tolist()
    metadatas = [metadata_from_json(value) for value in records["metadata_json"].astype(str).tolist()]
    store.add_texts(texts=texts, metadatas=metadatas, ids=ids)
    try:
        store.persist()  # type: ignore[attr-defined]
    except Exception:
        pass

    print(f"Indexed {len(texts)} records into {cfg.chroma_dir}")
    return len(texts)


if BUILD_CHROMA_INDEX:
    build_chroma_index(CFG, vector_records_df)
else:
    print("Chroma indexing is disabled. Set BUILD_CHROMA_INDEX = True to create embeddings.")

## **Search Example**

In [ ]:
def _open_vector_store(cfg: VectorConfig = CFG):
    embeddings = SentenceTransformerEmbeddings(model_name=cfg.embed_model)
    return Chroma(
        collection_name=cfg.chroma_collection,
        persist_directory=str(cfg.chroma_dir),
        embedding_function=embeddings,
    )


def _records_with_metadata(records: pd.DataFrame) -> pd.DataFrame:
    if records.empty:
        return records.copy()

    expanded = records.copy()
    metadata = pd.DataFrame([
        metadata_from_json(value)
        for value in expanded.get("metadata_json", pd.Series(dtype=str)).astype(str).tolist()
    ])
    for column in metadata.columns:
        if column not in expanded.columns:
            expanded[column] = metadata[column]
    return expanded


def _same_location(left: dict[str, Any], right: pd.Series, *, same_page: bool) -> bool:
    if left.get("source_file") != right.get("source_file"):
        return False
    if not same_page:
        return True

    left_page = left.get("page_num")
    right_page = right.get("page_num")
    if pd.isna(left_page) and pd.isna(right_page):
        return True
    return left_page == right_page


def _chunk_idx(value: Any) -> int | None:
    if value is None or pd.isna(value):
        return None
    try:
        return int(value)
    except Exception:
        return None


def _expanded_context(
    seed_metadata: dict[str, Any],
    records: pd.DataFrame,
    *,
    context: str,
    window: int,
    max_chars: int,
) -> str:
    if context == "chunk" or records.empty or seed_metadata.get("content_kind") != "text_chunk":
        return ""

    text_records = records[records.get("content_kind") == "text_chunk"].copy()
    if text_records.empty:
        return ""

    same_page = context == "window"
    candidates = text_records[
        text_records.apply(lambda row: _same_location(seed_metadata, row, same_page=same_page), axis=1)
    ].copy()
    if candidates.empty:
        return ""

    if context == "window":
        seed_idx = _chunk_idx(seed_metadata.get("chunk_idx"))
        if seed_idx is not None:
            candidates["_chunk_idx"] = candidates["chunk_idx"].apply(_chunk_idx)
            candidates = candidates[candidates["_chunk_idx"].between(seed_idx - window, seed_idx + window)]

    sort_cols = [col for col in ["page_num", "chunk_idx"] if col in candidates.columns]
    if sort_cols:
        candidates = candidates.sort_values(sort_cols, kind="stable")

    text = "\n\n".join(candidates["document"].astype(str).tolist())
    return text[:max_chars]


def semantic_search(
    query: str,
    k: int = 5,
    *,
    context: str = "chunk",
    window: int = 1,
    fetch_k: int | None = None,
    max_chars: int = 6000,
) -> list[dict[str, Any]]:
    """Retrieve semantic hits, optionally expanding each hit into broader context.

    context="chunk" returns the ranked chunks exactly as stored in Chroma.
    context="window" returns each hit plus neighboring chunks from the same page/document location.
    context="document" returns one bundled result per source document, seeded by the best hits.
    """
    if Chroma is None or SentenceTransformerEmbeddings is None:
        print("Chroma search is unavailable. Build the index after installing Chroma and embeddings support.")
        return []
    if context not in {"chunk", "window", "document"}:
        raise ValueError('context must be one of: "chunk", "window", "document"')

    store = _open_vector_store(CFG)
    seed_count = fetch_k or (k if context == "chunk" else max(k * 4, k + 5))
    seeds = store.similarity_search_with_score(query, k=seed_count)
    records = _records_with_metadata(vector_records_df)

    ranked: list[dict[str, Any]] = []
    seen_documents: set[str] = set()
    for seed_rank, (doc, score) in enumerate(seeds, start=1):
        metadata = dict(doc.metadata)
        source_file = str(metadata.get("source_file", ""))
        if context == "document" and source_file:
            if source_file in seen_documents:
                continue
            seen_documents.add(source_file)

        expanded = _expanded_context(
            metadata,
            records,
            context=context,
            window=window,
            max_chars=max_chars,
        )
        evidence = expanded or doc.page_content[:max_chars]
        ranked.append({
            "rank": len(ranked) + 1,
            "seed_rank": seed_rank,
            "score": float(score),
            "context_mode": context,
            "preview": doc.page_content[:500],
            "evidence": evidence,
            "metadata": metadata,
        })
        if len(ranked) >= k:
            break

    return ranked

In [ ]:
# Precise chunk hits:
#semantic_search("What drove revenue growth for Microsoft?", k=5)

In [ ]:
# Hit plus surrounding chunks when the answer spans a paragraph or adjacent chunks:
#semantic_search("What drove revenue growth for Microsoft?", k=5, context="window", window=2)

In [ ]:
# One expanded evidence bundle per source document when the answer may be document-level or cross-document:
#semantic_search("What drove revenue growth for Microsoft?", k=5, context="document", fetch_k=30)